# A Cholera Outbreak in London

## How to Use This Notebook

You've just watched your instructor work through the **airborne (sewer)** analysis. Now it's your turn: with your partner, recreate that analysis and then apply the same steps to the **Broad Street Pump**, at your own pace.

**Start here:** run the setup cell (below, in "The Data") that imports the libraries and loads the data.

As you go:
- The sewer cells were demonstrated — you'll **recreate them yourself**, then repeat the steps for the pump.
- Wherever you see `# YOUR CODE HERE`, it's your turn to write the code.
- Check `*Hint:*` notes when you and your partner get stuck, and the **Reference Card** at the end.

## Teaching Notes: 10-Minute Launch

*(Notes for you, the instructor. Students never see this notebook.)*

**Set up two screens.** Put the **student notebook** on the projector, and keep **this instructor notebook open on a second device** (a second laptop, a tablet, a printout, or a second monitor) as your private reference. This notebook shows the **completed code** for every demo cell plus a short **"Building it together"** note on what to say. **You don't need to memorize anything** — read the line here and type it into the projected student notebook.

Set the scene first with the story cells (the outbreak, the two theories, John Snow, the map, the pump, the sewers) — just a few minutes.

**During the demo (~10 minutes):** type these five cells live into the projected student notebook, talking through your thinking as you go:

1. `sewer_proximity` — label each house near/far from a sewer with `np.where`.
2. `death_status` — label each house Deaths/Non-Deaths with `np.where`.
3. `sewer_table` — build the contingency table with `pd.crosstab`.
4. Normalized table — the same table as column percentages.
5. Chi-squared p-value — `chi2_contingency` returns four values; keep the p-value (~**0.42**).

Land the point: a p-value of 0.42 means the sewer (airborne) link could easily be chance.

**Tip:** do a quick run-through on your own before class — it makes the live typing feel easy.

**Then release the class.** Students recreate the sewer cells and then run the **waterborne (pump)** analysis themselves — the same steps for the Broad Street Pump. They'll find a tiny p-value (**highly significant**), supporting John Snow's water theory over the airborne theory.

**As you circulate:** this notebook has the worked solutions for the pump cells (the `###` markers) — your answer key. Use the **Think About It** prompts to draw out the comparison between the two hypotheses, and point fast finishers to the **Challenge**.

## The Outbreak

In the late summer of 1849, a particularly bad outbreak of cholera struck the Soho neighborhood in central London. Between August 31 and September 10, over 500 people had died. By the end of the outbreak, the death toll was 616.

In this notebook, we will test two proposed explanations (hypotheses) of how cholera spread in London: through the **air** and through the **water**. That is, we will show that some hypotheses are likely a better fit for the data and are harder to reject, in a ***statistically significant*** way, than others. 

<br>

<table><tr>
    <td> <img src="https://github.com/jdomyancich/big-data-camp/blob/main/imgs/king_cholera.png?raw=true" alt="Drawing" style="width: 600px;"/> </td>
</tr></table>

<br>

## The Theories

Two predominant theories on the cause of cholera existed at the the time:

### Airborne

<img align="right" width="300" height="300" src='https://github.com/jdomyancich/big-data-camp/blob/main/imgs/airborne.png?raw=true'> 

Inhalation of a poison given off by dead or contaminated organic matter like sewage which enters the body through the lungs and poisons the blood. 

<img align="right" width="300" height="300" src='https://github.com/jdomyancich/big-data-camp/blob/main/imgs/waterborne.png?raw=true'>

### Waterborne

Ingestion of “excretions of the sick” which contain a living organism which infects the gastrointestinal system.

<img align="right" width="300" height="300" src='https://github.com/jdomyancich/big-data-camp/blob/main/imgs/john_snow.jpg?raw=true'>

## The Doctor

* John Snow
* Known for pioneering anesthesia techniques
* Noticed that cholera affected the gastrointestinal system
* Hypothesized that contaminated drinking water was the cause of cholera
* Most people disagreed with him

<img align="right" height="300" width="500" src = 'https://github.com/jdomyancich/big-data-camp/blob/main/imgs/soho_map.jpeg?raw=true'>

## The Map

* John Snow collected data for each household, including the number of deaths from cholera, during the 1854 outbreak.
* Each death is represented as a black line.
* Multiple deaths in the same household appear stacked.

## The Pump

<img align="right" width="400" src = 'https://github.com/jdomyancich/big-data-camp/blob/main/imgs/broad_street.png?raw=true'>

* John Snow centered his map on a particular water pump that he suspected to be the source of the ourbreak.
* The pump was located on Broad Street.
* John Snow suspected the Broad Street Pump to be the source of the outbreak.


## The Sewers

<img align="right" width="400" src = 'https://github.com/jdomyancich/big-data-camp/blob/main/imgs/broad_sewers.png?raw=true'>

* However, the neighborhood also had many sewers.
* Sewers were thought to be a source of cholera by many supporters of the airborne theory.
* Sewers are represented as squares in the map to the right.




## The Data

People in charge of the city’s sewers went door-to-door in the Soho neighborhood to assess the claim that toxic fumes from its sewers were causing the deaths. We have digitized this data into a .csv file that has the following columns: 

- **house_ID:** unique indtifier for the house
- **deaths:** the total deaths in that particular house 
- **dis_sewers:** distance (in meters) from the house to the nearest sewer (1 meter = 3.3 feet)
- **dis_bspump:** distance (in meters) from the house to the Broad St. pump

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

house_data = pd.read_csv('https://raw.githubusercontent.com/jdomyancich/big-data-camp/refs/heads/main/data/deaths_by_house.csv')
house_data.head()

,house_ID,deaths,dis_sewers,dis_bspump
0,1,0,10.08,125.00
1,2,1,14.64,119.94
2,3,0,18.47,116.27
3,4,0,22.98,112.56
4,5,0,27.47,109.10


# Our Own Data Experiment

In the case of the airborne and waterborne theories, we can separate people into groups. The exposed group (people living near a sewer or the water pump) is often called an **impact** or **treatment group** while the unexposed group (people living far from a sewer or the water pump) is the **control group**. When testing the airborne theory, we will group people based on whether they lived near a sewer or not and whether they died of cholera or not. When testing the waterborne theory, we will group people based on whether they lived near a certain water pump or not and whether they died of cholera or not. 

This will result in four groups for each proposed explanation. We will place them in a 2x2 **contingency table** (also called a ***two-way table*** or ***crosstab***). We will have to test each explanation separately. In all, that means four contingency tables: an expected (null) and an observed table for each of the two hypotheses.

# The Airborne Hypothesis: Investigating the Sewers

Now that we've talked about how to set up our experiment, let's apply this to the cholera data! 

The first theory we will explore assumes that cholera is airborne and that people get infected by inhaling toxic fumes from localized sources. In this case, the source is fumes emitted from sewers. 

<center><img src = 'https://github.com/jdomyancich/big-data-camp/blob/main/imgs/sewer.jpeg?raw=true' width=400><center>

If this theory was true, then closer proximity to sewers would make it more likely to inhale the toxic air and contract cholera. For simplicity, let us assume **someone is 'close' to a sewer if they less than 40 feet (12.2 meters) from one** ... otherwise they are 'far'. Unfortunately, we don't have the total number of people in each house. That data was not collected.  Therefore, we will have to count houses instead of people.

A contingency table simply shows the total frequencies of each variable, with one variable appearing on each axis. It technically does not matter, but a common approach is to put the independent (explanatory) variable on the x-axis and the dependent (outcome) variable on the y-axis. While there are libraries to create contingency tables for us, we will build some ourselves in order to better understand it. Here is the contingency table for the airborne theory:

<img src="https://github.com/jdomyancich/big-data-camp/blob/main/imgs/sewers_observed.png?raw=true" style="width: 600px;"/>

### Building the Observed Contingency Table

We will now build a contingency table for what was actually observed during the outbreak using the following variable names. 

#### Using Conditional Logic: `np.where()`

This allows you to create values based on conditions. For example, we can categorize houses as near or far from a sewer and create a new column with this information. We will use 12.2 meters as the cutoff for the coindition:

**Your instructor works through this one with the class. When you work on your own, you'll write it here yourself.**

**Building it together**

- We need to sort houses into two groups: near a sewer vs. far.
- `np.where(condition, value_if_true, value_if_false)` labels each row based on a test.
- Our test: is `dis_sewers` greater than 12.2? If so, "Far from Sewer", otherwise "Near Sewer".
- Save it as a new column `sewer_proximity`, then show `.head()` to check.

In [ ]:
# Label each house 'Near Sewer' or 'Far from Sewer' using np.where (cutoff 12.2 m)
###
house_data['sewer_proximity'] = np.where(house_data['dis_sewers'] > 12.2, "Far from Sewer", "Near Sewer")
house_data.head()
###

We can use `np.where` to label whether a house had a death or not.

**Your instructor works through this one with the class. When you work on your own, you'll write it here yourself.**

**Building it together**

- Same tool, different question: did the house have any deaths?
- Test `deaths == 0`: if true label "Non-Deaths", otherwise "Deaths".
- Save it as a new column `death_status`.

In [ ]:
# Label each house 'Deaths' or 'Non-Deaths' using np.where
###
house_data['death_status'] = np.where(house_data['deaths'] == 0, "Non-Deaths", "Deaths")
house_data.head()
###

Now we have the data (the four categories) to build our contingency table. Contingency tables are also known as a **two-way table** or **crosstabulation**. Fortunately, Pandas has a function called `crosstab()` that will construct the table for us:

**Your instructor works through this one with the class. When you work on your own, you'll write it here yourself.**

**Building it together**

- Now count houses in each combination of the two labels.
- `pd.crosstab(rows, columns)` builds the two-way table for us.
- Put `death_status` on the rows and `sewer_proximity` on the columns; save it as `sewer_table`.

In [ ]:
# Build the contingency table with pd.crosstab; store it in sewer_table
###
sewer_table = pd.crosstab(house_data["death_status"], house_data["sewer_proximity"])
sewer_table
###

**Does there appear to be a significant difference in the incidence of deaths between houses that are near a sewer vs. far from a sewer?** 

We can get a better sense of the difference between the groups by calculating the `Deaths` and `Non-Deaths` as percentages using the `normalize` argument in the `crosstab` function.

**Your instructor works through this one with the class. When you work on your own, you'll write it here yourself.**

**Building it together**

- Raw counts are hard to compare when the groups are different sizes.
- Add `normalize='columns'` and multiply by 100 to turn each column into percentages.
- Ask the class: do the death percentages look very different near vs. far?

In [ ]:
# Rebuild the table as column percentages; store it in norm_sewer_table
###
norm_sewer_table = pd.crosstab(house_data["death_status"], house_data["sewer_proximity"], normalize='columns') * 100
norm_sewer_table
###

### Calculating the p-value

Even if there is a difference between the two groups of houses, is it large enough to support that living close to a sewer is associated with higher cholera rates and not just a difference caused by randomness? 

The method for testing statistical significance in contingency tables is called a "chi-squared ($Chi^2$) analysis". 

There is library called "SciPy" that has a function that will do the chi-squared analysis for us.

In [51]:
from scipy.stats import chi2_contingency

The `chi2_contingency` function returns 4 values. We are only interested in the p-value. When doing data science in Python, it is common convention to use `_` characters to mark variables whose values we don't need. 

**Your instructor works through this one with the class. When you work on your own, you'll write it here yourself.**

**Building it together**

- Even a small difference could be random. The chi-squared test gives us a p-value.
- `chi2_contingency(sewer_table)` returns four values; we only want the second (the p-value).
- Use `_` for the values we're ignoring, then print the p-value.
- A big p-value (here about 0.42) means the difference could easily be chance.

In [ ]:
# Run chi2_contingency on sewer_table and print the p-value
###
_, p_value, _, _ = chi2_contingency(sewer_table)
print(f"p-value: {p_value:.2f}")
###

#### Think About It

You just ran a chi-squared test on the sewer (airborne) data.

1. The p-value came out around **0.42**. In plain words, what are the chances the death-rate difference between houses near vs. far from a sewer is just random?
2. Does this result support or weaken the airborne (sewer) theory?

# The Waterborne Hypothesis: Investigating the Broad Street Pump

Next, we want to explore the theory that cholera was transmitted through contaminated water. At the time, John Snow guessed that the water of a particular pump, the Broad Street Pump (BSP, for short), might have carried pieces of poisonous sewage. Did the data support this hypothesis? 

<center><img src="https://github.com/jdomyancich/big-data-camp/blob/main/imgs/pump3.jpeg?raw=true" alt="Drawing" style="width: 300px;"/><center>

If this theory was true, then closer proximity to the Broad Street Pump would make it more likely to drink its contaminated water and contract cholera. For simplicity, let us assume **someone is 'close' to the Broad Street Pump if they are at most 140 meters from it**... otherwise they are 'far'.


<img src="https://github.com/jdomyancich/big-data-camp/blob/main/imgs/pumps_observed.png?raw=true" style="width: 600px;"/>

**Label each house as near or far from the Broad Street Pump.**

*Hint: use `np.where` like you did for the sewers, but with the `dis_bspump` column and a cutoff of 140 meters. Houses at most 140 m away are "Near the Pump".*

In [ ]:
# Label each house 'Near the Pump' or 'Far from the Pump' using np.where (cutoff 140 m)
###
house_data['pump_proximity'] = np.where(house_data['dis_bspump'] <= 140, "Near the Pump", "Far from the Pump")
house_data.head()
###

**Build the contingency table.**

*Hint: use `pd.crosstab` with `death_status` and `pump_proximity`, just like the sewer table.*

In [ ]:
# Build the contingency table with pd.crosstab; store it in pump_table
###
pump_table = pd.crosstab(house_data["death_status"], house_data["pump_proximity"])
pump_table
###

**Normalize the contingency table.**

*Hint: add `normalize='columns'` and multiply by 100 to get percentages. Store it in `norm_pump_table`.*

In [ ]:
# Rebuild the table as column percentages; store it in norm_pump_table
###
norm_pump_table = pd.crosstab(house_data["death_status"], house_data["pump_proximity"], normalize='columns') * 100
norm_pump_table
###

**Is the difference statistically significant?**

*Hint: run `chi2_contingency` on your counts table (`pump_table`, not the percentages) and print the p-value.*

In [ ]:
# Run chi2_contingency on pump_table and print the p-value
###
_, p_value, _, _ = chi2_contingency(pump_table)
print(f"p-value: {p_value:.10f}")
###

#### Think About It

Now compare your two tests.

1. What does the pump (waterborne) p-value tell you about the chance that the link between the Broad Street Pump and cholera deaths is just random?
2. Put the two p-values side by side — airborne (sewer) vs. waterborne (pump). Which hypothesis does the data support more strongly?
3. John Snow argued for the waterborne theory and most people disagreed with him. Based on your results, was he right?

> Write your answer here! 

## Challenge: Explore on Your Own

Finished early? Dig deeper into the data. Use the cells below, and ask another pair or the instructor if you get stuck.

1. **Move the line.** You used 12.2 m for sewers and 140 m for the pump. Try a different cutoff and re-run the test. How much does the p-value change? How sensitive is the conclusion to where you draw the line?
2. **Put the theories side by side.** Print both p-values together and describe, in a sentence, which hypothesis the data supports and by how much.
3. **Picture it.** Make a simple bar chart of the death percentages near vs. far for one of the hypotheses.

In [ ]:
# Challenge: try a different cutoff, or print both p-values side by side
# YOUR CODE HERE

In [ ]:
# Challenge: make a simple bar chart of death percentages near vs. far
# YOUR CODE HERE

## Reference Card

Stuck? Here are the key patterns from today, all in one place.

**Label rows by a condition** (creates a new column):
`house_data['sewer_proximity'] = np.where(house_data['dis_sewers'] > 12.2, 'Far from Sewer', 'Near Sewer')`

**Build a contingency table (counts):**
`sewer_table = pd.crosstab(house_data['death_status'], house_data['sewer_proximity'])`

**Show column percentages instead of counts:**
`pd.crosstab(rows, cols, normalize='columns') * 100`

**Chi-squared p-value** (we keep only the 2nd of the 4 return values):
`_, p_value, _, _ = chi2_contingency(sewer_table)`
`print(f"p-value: {p_value:.2f}")`

A small p-value (say, below 0.05) means the difference is unlikely to be due to chance.